# Librerías

In [44]:
import pandas as pd
import numpy as np
import re
import pathlib as Path
import unicodedata

# Documents

Extraer los datos de los últimos 5 años (2020 - 2024):
Hojas de formación en las empresas, contexto y errores muestreo
Extraemos todos los datos respecto a formación en las empresas pero después analizaremos solo (en el caso de algunas tablas de formación cogeremos ya las específicas de servicios [e.g. 2024 EAL-23 y 23c]):
CCAA: Total y Cataluña y otras CCAA donde tenemos sede (Madrid, Valencia, Sevilla y Bilbao)
Tamaño empresa: más de 499
Sector Transporte y almacenamiento y cuando agregado servicios

TABLAS CONTEXTO
2020, 2021, 2022, 2023, 2024:
EAL-C1, EAL-C2, EAL-C3

TABLAS FORMACIÓN EMPRESAS
2020, 2021, 2022:
EAL-16, EAL-17, EAL-18, EAL-19, EAL-20, EAL-21, EAL-22, EAL-24, EAL-25

2023, 2024 añadido:
EAL-18a, EAL-18b, EAL-18c

ERRORES MUESTREO
2020, 2021, 2022, 2023, 2024:
EAL-M1


He hagut de fer: pip install openpyxl (és l'engine de pandas per excel més nous)

In [45]:
import pandas as pd
from pathlib import Path

def encontrar_raiz_repo(inicio=None):
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if (candidata / "Equip_31").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del repositorio con Equip_31.")

REPO_ROOT = encontrar_raiz_repo()

# Ruta relativa dentro del repositorio Git.
file_path_rel = Path("Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx")
file_path = REPO_ROOT / file_path_rel

print("Raíz repo:", REPO_ROOT)
print("Excel origen:", file_path_rel)
print("Archivo existe:", file_path.exists())

if not file_path.exists():
    raise FileNotFoundError(f"No existe el Excel esperado: {file_path}")

sheets_to_extract = [
    # TABLAS CONTEXTO
    "EAL-C1", "EAL-C2", "EAL-C3",

    # TABLAS FORMACIÓN EMPRESAS
    "EAL-16", "EAL-17", "EAL-18", "EAL-19", "EAL-20", "EAL-21",
    "EAL-22", "EAL-23", "EAL-24", "EAL-25",
    "EAL-18a", "EAL-18b", "EAL-18c",

    # ERRORES MUESTREO
    "EAL-M1"
]

# Read selected sheets into a dictionary of DataFrames
dfs = pd.read_excel(
    file_path,
    sheet_name=sheets_to_extract,
    header=None,
    dtype=str
)

# Basic cleaning: remove fully empty rows and columns
for sheet_name, df in dfs.items():
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.reset_index(drop=True)
    dfs[sheet_name] = df


Raíz repo: /Users/fedeur/ProjecteData
Excel origen: Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx
Archivo existe: True


In [46]:
dfs["EAL-23"]

,0,1,2,3,4,5
0,ENCUESTA ANUAL LABORAL,NaN,NaN,NaN,NaN,EAL
1,,NaN,NaN,NaN,NaN,Volver al índice
2,EAL-23. EMPRESAS QUE PROPORCIONARON FORMACIÓN ...,NaN,NaN,NaN,NaN,NaN
3,Año 2024. Porcentaje sobre el total de empresa...,NaN,NaN,NaN,NaN,NaN
4,NaN,TOTAL,NADA,POCO,BASTANTE,MUCHO
5,Responder a un sistema predeterminado de promo...,100,24.833,42.321,26.86,5.987
6,Readaptar al personal a los cambios técnicos i...,100,12.615,22.193,49.421,15.771
7,Readaptar al personal a los cambios organizati...,100,14.246,29.411,44.323,12.02
8,Adaptar al personal recién incorporado a las t...,100,7.888,15.038,50.533,26.541
9,Mejorar la cualificación básica del personal p...,100,8.168,22.567,51.043,18.222


In [47]:
# Output folder dentro del repositorio Git.
output_folder_rel = Path("Equip_31/Data/Processed/EAL/2024")
output_folder = REPO_ROOT / output_folder_rel
output_folder.mkdir(parents=True, exist_ok=True)

# Save each dataframe as CSV
for sheet_name, df in dfs.items():
    output_path = output_folder / f"{sheet_name}.csv"

    df.to_csv(
        output_path,
        index=False,
        header=False,
        encoding="utf-8-sig"
    )

print(f"CSV files saved in: {output_folder_rel}")


CSV files saved in: Equip_31/Data/Processed/EAL/2024


## Normalización estructural de EAL-16

A partir de aquí se trabaja con el CSV extraído de la hoja `EAL-16`. Las rutas se resuelven desde la raíz del repositorio para que el notebook funcione aunque se ejecute desde otra carpeta.


In [48]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

from pathlib import Path
import re
import unicodedata
import pandas as pd

def encontrar_raiz_repo(inicio=None):
    """Devuelve la raíz del repositorio buscando la carpeta Equip_31."""
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if (candidata / "Equip_31").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del repositorio con Equip_31.")

REPO_ROOT = encontrar_raiz_repo()

# Ruta relativa dentro del repositorio Git.
archivo_csv_rel = Path("Equip_31/Data/Processed/EAL/2024/EAL-16.csv")
archivo_csv = REPO_ROOT / archivo_csv_rel

# Excel oficial raw, usado solo para comprobación y trazabilidad.
archivo_raw_rel = Path("Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx")
archivo_raw = REPO_ROOT / archivo_raw_rel

ANIO = 2024
HOJA = "EAL-16"

print("Raíz repo:", REPO_ROOT)
print("Ruta relativa:", archivo_csv_rel)
print("CSV existe:", archivo_csv.exists())
print("CSV resuelto:", archivo_csv)
print("Raw existe:", archivo_raw.exists())
print("Raw resuelto:", archivo_raw)

if not archivo_csv.exists():
    raise FileNotFoundError(f"No existe el CSV esperado: {archivo_csv}")
if not archivo_raw.exists():
    raise FileNotFoundError(f"No existe el Excel raw esperado: {archivo_raw}")


Raíz repo: /Users/fedeur/ProjecteData
Ruta relativa: Equip_31/Data/Processed/EAL/2024/EAL-16.csv
CSV existe: True
CSV resuelto: /Users/fedeur/ProjecteData/Equip_31/Data/Processed/EAL/2024/EAL-16.csv
Raw existe: True
Raw resuelto: /Users/fedeur/ProjecteData/Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx


## Funciones auxiliares

Estas funciones limpian texto, detectan las subtablas internas de `EAL-16` y asignan metadatos estructurales: ámbito, sector, tamaño de empresa y CCAA.


In [49]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def limpiar_texto(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).replace("\n", " ")
    valor = re.sub(r"\s+", " ", valor)
    return valor.strip()

def quitar_acentos(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().strip()

def extraer_codigo_subtabla(texto):
    match = re.match(r"^(EAL-16[a-f]?)\.", limpiar_texto(texto), flags=re.IGNORECASE)
    return match.group(1).upper() if match else None

def metadatos_subtabla(codigo, titulo):
    """
    Asigna solo las dimensiones que realmente desagrega cada subtabla.

    Regla metodológica para EAL-16:
    - EAL-16: total empresas; no desagrega por sector, tamaño ni CCAA.
    - EAL-16a/b/c: desagrega por tamaño de empresa; sector y CCAA no aplican.
    - EAL-16d/e/f: desagrega por sector agregado; tamaño y CCAA no aplican.
    """
    metadatos = {
        "Ámbito": "Total empresas",
        "Sector": "",
        "Tamaño Empresa": "",
        "CCAA": "",
    }

    if codigo == "EAL-16A":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = "5 a 49 trabajadores"
    elif codigo == "EAL-16B":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = "50 a 499 trabajadores"
    elif codigo == "EAL-16C":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = "Más de 499 trabajadores"
    elif codigo == "EAL-16D":
        metadatos["Ámbito"] = "Sector agregado"
        metadatos["Sector"] = "Industria"
    elif codigo == "EAL-16E":
        metadatos["Ámbito"] = "Sector agregado"
        metadatos["Sector"] = "Construcción"
    elif codigo == "EAL-16F":
        metadatos["Ámbito"] = "Sector agregado"
        metadatos["Sector"] = "Servicios"

    return metadatos

def fila_es_cabecera(row):
    valores = [limpiar_texto(x).upper() for x in row.tolist()]
    return {"TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"}.issubset(set(valores))


## Lectura del CSV original

Se lee el CSV procesado por la extracción previa, sin asumir encabezado fijo.


In [50]:
# ============================================================
# LECTURA DEL CSV ORIGINAL
# ============================================================

df_raw = pd.read_csv(
    archivo_csv,
    header=None,
    dtype=str,
    keep_default_na=False
)

df_raw = df_raw.map(limpiar_texto)
df_raw = df_raw.replace("", pd.NA)
df_raw = df_raw.dropna(how="all")
df_raw = df_raw.dropna(axis=1, how="all")
df_raw = df_raw.fillna("").reset_index(drop=True)

print("Dimensión original:", df_raw.shape)
display(df_raw.head(20))


Dimensión original: (93, 6)


,0,1,2,3,4,5
0,ENCUESTA ANUAL LABORAL,,,,,EAL
1,,,,,,Volver al índice
2,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,,,,,
3,Año 2024. Porcentaje sobre el total de empresas.,,,,,
4,,TOTAL,NADA,POCO,BASTANTE,MUCHO
5,De dirección,100,10.414,18.818,40.283,30.484
6,De trabajo en equipo,100,2.224,5.822,45.125,46.829
7,De atención al público/ trato a clientes,100,5.128,13.392,37.144,44.336
8,Administrativas de oficina,100,11.117,25.313,37.788,25.782
9,De resolución de problemas (localización de pr...,100,5.893,15.752,44.063,34.292


## Conversión de EAL-16 a formato largo

`EAL-16.csv` contiene varias subtablas apiladas: total, tamaños de empresa y sectores. El parser detecta cada bloque y convierte las columnas `TOTAL`, `NADA`, `POCO`, `BASTANTE` y `MUCHO` en la columna `Variable`.


In [51]:
# ============================================================
# PARSEAR SUBTABLAS Y PASAR A FORMATO LARGO
# ============================================================

registros = []
titulo_actual = None
codigo_actual = None
variables_actuales = None
metadatos_actuales = None

for _, row in df_raw.iterrows():
    primera_celda = limpiar_texto(row.iloc[0])
    codigo_detectado = extraer_codigo_subtabla(primera_celda)

    if codigo_detectado:
        codigo_actual = codigo_detectado
        titulo_actual = primera_celda
        variables_actuales = None
        metadatos_actuales = metadatos_subtabla(codigo_actual, titulo_actual)
        continue

    if codigo_actual and fila_es_cabecera(row):
        variables_actuales = [limpiar_texto(x).upper() for x in row.tolist()]
        continue

    if not (codigo_actual and variables_actuales):
        continue

    categoria = primera_celda
    if categoria == "":
        continue

    for posicion, variable in enumerate(variables_actuales):
        if posicion == 0 or variable == "":
            continue

        valor = limpiar_texto(row.iloc[posicion]) if posicion < len(row) else ""
        if valor == "":
            continue

        registros.append({
            "Año": ANIO,
            "Hoja": HOJA,
            "Título de tabla": titulo_actual,
            "Categoría": categoria,
            "Variable": variable,
            "Valor": valor,
            "Ámbito": metadatos_actuales["Ámbito"],
            "Sector": metadatos_actuales["Sector"],
            "Tamaño Empresa": metadatos_actuales["Tamaño Empresa"],
            "CCAA": metadatos_actuales["CCAA"],
        })

df_largo = pd.DataFrame(registros)
df_largo["Valor"] = pd.to_numeric(df_largo["Valor"], errors="coerce")

columnas_finales = [
    "Año",
    "Hoja",
    "Título de tabla",
    "Categoría",
    "Variable",
    "Valor",
    "Ámbito",
    "Sector",
    "Tamaño Empresa",
    "CCAA",
]

df_largo = df_largo[columnas_finales]

print("Dimensión final:", df_largo.shape)
print("Subtablas detectadas:", df_largo["Título de tabla"].nunique())
print("Categorías detectadas:", df_largo["Categoría"].nunique())
display(df_largo.head(50))


Dimensión final: (350, 10)
Subtablas detectadas: 7
Categorías detectadas: 10


,Año,Hoja,Título de tabla,Categoría,Variable,Valor,Ámbito,Sector,Tamaño Empresa,CCAA
0,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,TOTAL,100.000,Total empresas,,,
1,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,NADA,10.414,Total empresas,,,
2,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,POCO,18.818,Total empresas,,,
3,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,BASTANTE,40.283,Total empresas,,,
4,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,MUCHO,30.484,Total empresas,,,
5,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,TOTAL,100.000,Total empresas,,,
6,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,NADA,2.224,Total empresas,,,
7,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,POCO,5.822,Total empresas,,,
8,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,BASTANTE,45.125,Total empresas,,,
9,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,MUCHO,46.829,Total empresas,,,


## Validación interna y visualización completa de EAL-16

Se comprueban dimensiones y conteo esperado, y después se muestra completo `df_largo` para revisión manual.


In [52]:
# ============================================================
# VALIDACIÓN INTERNA Y VISUALIZACIÓN COMPLETA DE EAL-16
# ============================================================

columnas_esperadas = [
    "Año", "Hoja", "Título de tabla", "Categoría", "Variable",
    "Valor", "Ámbito", "Sector", "Tamaño Empresa", "CCAA"
]

assert list(df_largo.columns) == columnas_esperadas
assert df_largo.shape == (350, 10)
assert not df_largo.empty
assert df_largo["Valor"].notna().all()

print("EAL-16 validado internamente")
print("Forma df_largo:", df_largo.shape)
print("Cálculo esperado: 7 bloques x 10 competencias x 5 variables = 350 registros")

print()
print("Resumen para revisar contra el Excel raw:")
display(
    df_largo.groupby(["Ámbito", "Sector", "Tamaño Empresa", "CCAA"], dropna=False)
    .size()
    .reset_index(name="n_registros")
)

print()
print("df_largo completo:")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_largo)


EAL-16 validado internamente
Forma df_largo: (350, 10)
Cálculo esperado: 7 bloques x 10 competencias x 5 variables = 350 registros

Resumen para revisar contra el Excel raw:


,Ámbito,Sector,Tamaño Empresa,CCAA,n_registros
0,Sector agregado,Construcción,,,50
1,Sector agregado,Industria,,,50
2,Sector agregado,Servicios,,,50
3,Tamaño empresa,,5 a 49 trabajadores,,50
4,Tamaño empresa,,50 a 499 trabajadores,,50
5,Tamaño empresa,,Más de 499 trabajadores,,50
6,Total empresas,,,,50



df_largo completo:


,Año,Hoja,Título de tabla,Categoría,Variable,Valor,Ámbito,Sector,Tamaño Empresa,CCAA
0,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De dirección,TOTAL,100.000,Total empresas,,,
1,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De dirección,NADA,10.414,Total empresas,,,
2,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De dirección,POCO,18.818,Total empresas,,,
3,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De dirección,BASTANTE,40.283,Total empresas,,,
4,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De dirección,MUCHO,30.484,Total empresas,,,
5,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De trabajo en equipo,TOTAL,100.000,Total empresas,,,
6,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De trabajo en equipo,NADA,2.224,Total empresas,,,
7,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De trabajo en equipo,POCO,5.822,Total empresas,,,
8,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De trabajo en equipo,BASTANTE,45.125,Total empresas,,,
9,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De trabajo en equipo,MUCHO,46.829,Total empresas,,,


In [53]:
pd.set_option("display.max_rows", None)

display(df_largo)

,Año,Hoja,Título de tabla,Categoría,Variable,Valor,Ámbito,Sector,Tamaño Empresa,CCAA
0,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,TOTAL,100.000,Total empresas,,,
1,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,NADA,10.414,Total empresas,,,
2,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,POCO,18.818,Total empresas,,,
3,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,BASTANTE,40.283,Total empresas,,,
4,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De dirección,MUCHO,30.484,Total empresas,,,
5,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,TOTAL,100.000,Total empresas,,,
6,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,NADA,2.224,Total empresas,,,
7,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,POCO,5.822,Total empresas,,,
8,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,BASTANTE,45.125,Total empresas,,,
9,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,De trabajo en equipo,MUCHO,46.829,Total empresas,,,


## Normalización estructural de EAL-17 y EAL-18

Estas dos hojas tienen una estructura distinta a `EAL-16`: no son subtablas apiladas, sino una tabla con filas agrupadas por `TOTAL`, `TAMAÑO DE LA EMPRESA`, `ACTIVIDAD ECONÓMICA` y `COMUNIDAD AUTÓNOMA`. Se convierten a formato largo sin cruzar dimensiones que no aparecen cruzadas en el raw.

In [54]:
# ============================================================
# PARSER GENERAL PARA EAL-17 Y EAL-18
# ============================================================

SECCIONES_FILA = {
    "TAMAÑO DE LA EMPRESA": "Tamaño empresa",
    "ACTIVIDAD ECONÓMICA": "Sector",
    "COMUNIDAD AUTÓNOMA": "CCAA",
}


def leer_csv_hoja(hoja):
    ruta_rel = Path(f"Equip_31/Data/Processed/EAL/2024/{hoja}.csv")
    ruta = REPO_ROOT / ruta_rel
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el CSV esperado: {ruta}")

    df = pd.read_csv(ruta, header=None, dtype=str, keep_default_na=False)
    df = df.map(limpiar_texto)
    df = df.replace("", pd.NA)
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.fillna("").reset_index(drop=True)
    return df, ruta_rel


def metadatos_fila(categoria, seccion_actual):
    metadatos = {
        "Ámbito": "Total empresas",
        "Sector": "",
        "Tamaño Empresa": "",
        "CCAA": "",
    }

    if categoria.upper() == "TOTAL":
        return metadatos

    if seccion_actual == "Tamaño empresa":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = categoria
    elif seccion_actual == "Sector":
        metadatos["Ámbito"] = "Sector"
        metadatos["Sector"] = categoria
    elif seccion_actual == "CCAA":
        metadatos["Ámbito"] = "CCAA"
        metadatos["CCAA"] = categoria

    return metadatos


def detectar_titulo_csv(df, hoja):
    for valor in df.iloc[:, 0].tolist():
        texto = limpiar_texto(valor)
        if texto.upper().startswith(f"{hoja}."):
            return texto
    return ""


def construir_variables_eal17(df):
    # En el CSV limpio, las cabeceras son las filas 4 y 5: grupo superior + subvariable.
    grupo_superior = [limpiar_texto(x) for x in df.iloc[4].tolist()]
    subvariable = [limpiar_texto(x) for x in df.iloc[5].tolist()]

    variables = {}
    grupo_actual = ""
    for col in range(1, df.shape[1]):
        if grupo_superior[col]:
            grupo_actual = grupo_superior[col]
        sub = subvariable[col]

        if col == 1:
            variables[col] = "TOTAL"
        elif grupo_actual and sub:
            variables[col] = f"{grupo_actual} - {sub}"
        elif grupo_actual:
            variables[col] = grupo_actual
        else:
            variables[col] = sub
    return variables, 6


def construir_variables_eal18(df):
    # En el CSV limpio, las cabeceras son las filas 4 y 5: grupo superior + subvariable.
    grupo_superior = [limpiar_texto(x) for x in df.iloc[4].tolist()]
    subvariable = [limpiar_texto(x) for x in df.iloc[5].tolist()]

    variables = {}
    grupo_actual = ""
    for col in range(1, df.shape[1]):
        if grupo_superior[col]:
            grupo_actual = grupo_superior[col]
        sub = subvariable[col]
        variables[col] = f"{grupo_actual} - {sub}" if grupo_actual and sub else (grupo_actual or sub)
    return variables, 6


def parsear_eal_17_18(hoja, constructor_variables):
    df, ruta_rel = leer_csv_hoja(hoja)
    titulo = detectar_titulo_csv(df, hoja)
    variables, fila_inicio_datos = constructor_variables(df)

    registros = []
    seccion_actual = None

    for _, row in df.iloc[fila_inicio_datos:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "":
            continue

        categoria_norm = categoria.upper()
        if categoria_norm in SECCIONES_FILA:
            seccion_actual = SECCIONES_FILA[categoria_norm]
            continue
        if categoria.startswith("("):
            continue

        metadatos = metadatos_fila(categoria, seccion_actual)

        for col, variable in variables.items():
            valor = limpiar_texto(row.iloc[col]) if col < len(row) else ""
            if valor == "":
                continue

            registros.append({
                "Año": ANIO,
                "Hoja": hoja,
                "Título de tabla": titulo,
                "Categoría": categoria,
                "Variable": variable,
                "Valor": pd.to_numeric(valor, errors="coerce"),
                "Ámbito": metadatos["Ámbito"],
                "Sector": metadatos["Sector"],
                "Tamaño Empresa": metadatos["Tamaño Empresa"],
                "CCAA": metadatos["CCAA"],
            })

    df_largo_hoja = pd.DataFrame(registros)[columnas_finales]
    print(f"{hoja} desde {ruta_rel}: {df.shape} -> {df_largo_hoja.shape}")
    return df_largo_hoja


## Dataframe largo de EAL-17

`EAL-17` indica si las empresas impartieron formación, con cortes por total, tamaño de empresa, actividad económica y comunidad autónoma.

In [55]:
# ============================================================
# EAL-17 A FORMATO LARGO
# ============================================================

df_eal17_largo = parsear_eal_17_18("EAL-17", construir_variables_eal17)

display(df_eal17_largo.head(40))
display(
    df_eal17_largo.groupby(["Ámbito", "Sector", "Tamaño Empresa", "CCAA"], dropna=False)
    .size()
    .reset_index(name="n_registros")
    .head(40)
)


EAL-17 desde Equip_31/Data/Processed/EAL/2024/EAL-17.csv: (42, 7) -> (192, 10)


,Año,Hoja,Título de tabla,Categoría,Variable,Valor,Ámbito,Sector,Tamaño Empresa,CCAA
0,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,TOTAL,TOTAL,100.000,Total empresas,,,
1,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Total,73.468,Total empresas,,,
2,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Impartic...,60.220,Total empresas,,,
3,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo imp...,25.366,Total empresas,,,
4,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo otr...,14.415,Total empresas,,,
5,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,TOTAL,EMPRESAS QUE NO PROPORCIONAN FORMACIÓN,26.532,Total empresas,,,
6,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,De 5 a 9 trabajadores,TOTAL,100.000,Tamaño empresa,,De 5 a 9 trabajadores,
7,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,De 5 a 9 trabajadores,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Total,65.319,Tamaño empresa,,De 5 a 9 trabajadores,
8,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,De 5 a 9 trabajadores,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Impartic...,51.321,Tamaño empresa,,De 5 a 9 trabajadores,
9,2024,EAL-17,EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓ...,De 5 a 9 trabajadores,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo imp...,28.014,Tamaño empresa,,De 5 a 9 trabajadores,


,Ámbito,Sector,Tamaño Empresa,CCAA,n_registros
0,CCAA,,,Andalucía,6
1,CCAA,,,Aragón,6
2,CCAA,,,Asturias (Principado de),6
3,CCAA,,,Balears (Illes),6
4,CCAA,,,Canarias,6
5,CCAA,,,Cantabria,6
6,CCAA,,,Castilla y León,6
7,CCAA,,,Castilla-La Mancha,6
8,CCAA,,,Cataluña,6
9,CCAA,,,Comunitat Valenciana,6


## Dataframe largo de EAL-18

`EAL-18` relaciona detección de necesidades formativas con impartición o no de formación, con los mismos cortes por total, tamaño de empresa, actividad económica y comunidad autónoma.

In [56]:
# ============================================================
# EAL-18 A FORMATO LARGO
# ============================================================

df_eal18_largo = parsear_eal_17_18("EAL-18", construir_variables_eal18)

display(df_eal18_largo.head(40))
display(
    df_eal18_largo.groupby(["Ámbito", "Sector", "Tamaño Empresa", "CCAA"], dropna=False)
    .size()
    .reset_index(name="n_registros")
    .head(40)
)


EAL-18 desde Equip_31/Data/Processed/EAL/2024/EAL-18.csv: (43, 7) -> (192, 10)


,Año,Hoja,Título de tabla,Categoría,Variable,Valor,Ámbito,Sector,Tamaño Empresa,CCAA
0,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSON...,29.123694,Total empresas,,,
1,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSON...,89.974511,Total empresas,,,
2,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSON...,10.025489,Total empresas,,,
3,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PER...,70.876306,Total empresas,,,
4,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PER...,66.570590,Total empresas,,,
5,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PER...,33.429410,Total empresas,,,
6,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,De 5 a 9 trabajadores,DETECTARON NECESIDADES FORMATIVAS DE SU PERSON...,23.671862,Tamaño empresa,,De 5 a 9 trabajadores,
7,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,De 5 a 9 trabajadores,DETECTARON NECESIDADES FORMATIVAS DE SU PERSON...,83.544726,Tamaño empresa,,De 5 a 9 trabajadores,
8,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,De 5 a 9 trabajadores,DETECTARON NECESIDADES FORMATIVAS DE SU PERSON...,16.455274,Tamaño empresa,,De 5 a 9 trabajadores,
9,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESID...,De 5 a 9 trabajadores,NO DETECTARON NECESIDADES FORMATIVAS DE SU PER...,76.328138,Tamaño empresa,,De 5 a 9 trabajadores,


,Ámbito,Sector,Tamaño Empresa,CCAA,n_registros
0,CCAA,,,Andalucía,6
1,CCAA,,,Aragón,6
2,CCAA,,,Asturias (Principado de),6
3,CCAA,,,Balears (Illes),6
4,CCAA,,,Canarias,6
5,CCAA,,,Cantabria,6
6,CCAA,,,Castilla y León,6
7,CCAA,,,Castilla-La Mancha,6
8,CCAA,,,Cataluña,6
9,CCAA,,,Comunitat Valenciana,6


## Validación interna y visualización completa de EAL-17 y EAL-18

Se comprueban dimensiones y conteos esperados, y después se muestran completos los dataframes para revisión manual contra el Excel raw si hace falta.


In [57]:
# ============================================================
# VALIDACIÓN INTERNA Y VISUALIZACIÓN COMPLETA DE EAL-17 Y EAL-18
# ============================================================

assert df_eal17_largo.shape == (192, 10)
assert df_eal18_largo.shape == (192, 10)
assert list(df_eal17_largo.columns) == columnas_finales
assert list(df_eal18_largo.columns) == columnas_finales
assert df_eal17_largo["Valor"].notna().all()
assert df_eal18_largo["Valor"].notna().all()

print("EAL-17 y EAL-18 validados internamente")
print("EAL-17: 32 filas de categorías/desgloses x 6 variables = 192 registros")
print("EAL-18: 32 filas de categorías/desgloses x 6 variables = 192 registros")

print()
print("Resumen EAL-17 para revisar contra el Excel raw:")
display(
    df_eal17_largo.groupby(["Ámbito", "Sector", "Tamaño Empresa", "CCAA"], dropna=False)
    .size()
    .reset_index(name="n_registros")
)

print()
print("df_eal17_largo completo:")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal17_largo)

print()
print("Resumen EAL-18 para revisar contra el Excel raw:")
display(
    df_eal18_largo.groupby(["Ámbito", "Sector", "Tamaño Empresa", "CCAA"], dropna=False)
    .size()
    .reset_index(name="n_registros")
)

print()
print("df_eal18_largo completo:")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal18_largo)


EAL-17 y EAL-18 validados internamente
EAL-17: 32 filas de categorías/desgloses x 6 variables = 192 registros
EAL-18: 32 filas de categorías/desgloses x 6 variables = 192 registros

Resumen EAL-17 para revisar contra el Excel raw:


,Ámbito,Sector,Tamaño Empresa,CCAA,n_registros
0,CCAA,,,Andalucía,6
1,CCAA,,,Aragón,6
2,CCAA,,,Asturias (Principado de),6
3,CCAA,,,Balears (Illes),6
4,CCAA,,,Canarias,6
5,CCAA,,,Cantabria,6
6,CCAA,,,Castilla y León,6
7,CCAA,,,Castilla-La Mancha,6
8,CCAA,,,Cataluña,6
9,CCAA,,,Comunitat Valenciana,6



df_eal17_largo completo:


,Año,Hoja,Título de tabla,Categoría,Variable,Valor,Ámbito,Sector,Tamaño Empresa,CCAA
0,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",TOTAL,TOTAL,100.000,Total empresas,,,
1,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Total,73.468,Total empresas,,,
2,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Impartición de cursos y otros tipos de formación (1),60.220,Total empresas,,,
3,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo impartición de cursos (1),25.366,Total empresas,,,
4,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo otros tipos de formación (1),14.415,Total empresas,,,
5,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",TOTAL,EMPRESAS QUE NO PROPORCIONAN FORMACIÓN,26.532,Total empresas,,,
6,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 5 a 9 trabajadores,TOTAL,100.000,Tamaño empresa,,De 5 a 9 trabajadores,
7,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 5 a 9 trabajadores,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Total,65.319,Tamaño empresa,,De 5 a 9 trabajadores,
8,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 5 a 9 trabajadores,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Impartición de cursos y otros tipos de formación (1),51.321,Tamaño empresa,,De 5 a 9 trabajadores,
9,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 5 a 9 trabajadores,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo impartición de cursos (1),28.014,Tamaño empresa,,De 5 a 9 trabajadores,



Resumen EAL-18 para revisar contra el Excel raw:


,Ámbito,Sector,Tamaño Empresa,CCAA,n_registros
0,CCAA,,,Andalucía,6
1,CCAA,,,Aragón,6
2,CCAA,,,Asturias (Principado de),6
3,CCAA,,,Balears (Illes),6
4,CCAA,,,Canarias,6
5,CCAA,,,Cantabria,6
6,CCAA,,,Castilla y León,6
7,CCAA,,,Castilla-La Mancha,6
8,CCAA,,,Cataluña,6
9,CCAA,,,Comunitat Valenciana,6



df_eal18_largo completo:


,Año,Hoja,Título de tabla,Categoría,Variable,Valor,Ámbito,Sector,Tamaño Empresa,CCAA
0,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Total,29.123694,Total empresas,,,
1,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Proporcionaron formación (1),89.974511,Total empresas,,,
2,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,TOTAL,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - No proporcionaron formación (1),10.025489,Total empresas,,,
3,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Total,70.876306,Total empresas,,,
4,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Proporcionaron formación (2),66.570590,Total empresas,,,
5,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,TOTAL,NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - No proporcionaron formación (2),33.429410,Total empresas,,,
6,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 5 a 9 trabajadores,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Total,23.671862,Tamaño empresa,,De 5 a 9 trabajadores,
7,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 5 a 9 trabajadores,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Proporcionaron formación (1),83.544726,Tamaño empresa,,De 5 a 9 trabajadores,
8,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 5 a 9 trabajadores,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - No proporcionaron formación (1),16.455274,Tamaño empresa,,De 5 a 9 trabajadores,
9,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 5 a 9 trabajadores,NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Total,76.328138,Tamaño empresa,,De 5 a 9 trabajadores,
